In [71]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import time

In [72]:
spark = SparkSession.builder \
    .appName("Ecommerce_Lab") \
    .master("spark://spark-master:7077") \
    .config("spark.ui.port", "4042") \
    .config("spark.executor.instances","2") \
    .config("spark.executor.cores","1") \
    .config("spark.executor.memory","1g") \
    .getOrCreate()

In [73]:
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/opt/spark/work-dir/raw_data/ecommerce_orders.csv")

In [74]:
df.count()

5050

In [46]:
df.show()

[Stage 1:>                                                          (0 + 1) / 1]

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+
|order_id|customer_id|product_id|   category| price|quantity|order_status|order_date|payment_method|country|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+
|       1|         52|       380|       Home|643.03|     3.0|     Success|2026-03-12| Bank Transfer|     US|
|       2|        448|       120|     Beauty|  39.5|     9.0|     Success|2026-03-12| Bank Transfer| Canada|
|       3|       1781|         4|    Fashion|285.41|     6.0|      FAILED|2025-01-09| Bank Transfer|     US|
|       4|        705|       310|       Home|365.39|     8.0|     success|2023-07-31|        PayPal| Canada|
|       5|        143|        24|      Books|200.37|     5.0|      FAILED|2023-09-14| Bank Transfer|  Japan|
|       6|        334|       190|       Home|835.77|     5.0|     Success|2023-07-16|    Debit Card| Canada|
|       7|        3

In [47]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country: string (nullable = true)



In [75]:
df.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1)  \
    .count()

50

In [76]:
df.select([
    sum(
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

+--------+-----------+----------+--------+-----+--------+------------+----------+--------------+-------+
|order_id|customer_id|product_id|category|price|quantity|order_status|order_date|payment_method|country|
+--------+-----------+----------+--------+-----+--------+------------+----------+--------------+-------+
|       0|          0|         0|       0|    0|     517|           0|         0|             0|      0|
+--------+-----------+----------+--------+-----+--------+------------+----------+--------------+-------+



In [77]:
df.filter(col("quantity").isNull()).count()

517

In [78]:
df_check_date = df.withColumn(
    "parsed_order_date",
    to_date(col("order_date"), "yyyy-MM-dd")
)
df_check_date = df_check_date.filter(
    col("order_date").isNotNull() & col("parsed_order_date").isNull()
)

In [79]:
df_check_date.show()

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-----------------+
|order_id|customer_id|product_id|   category| price|quantity|order_status|order_date|payment_method|country|parsed_order_date|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-----------------+
|      54|        692|       390|     Sports|460.15|     7.0|     SUCCESS|  bad_date|   Credit Card|Germany|             NULL|
|      81|         60|       499|      Books|489.86|     4.0|     PENDING|  bad_date| Bank Transfer|     US|             NULL|
|     117|        879|       433|     Beauty| 233.9|    10.0|     Success|  bad_date|        PayPal|     US|             NULL|
|     122|        997|       270|      Books| 681.7|     4.0|      failed|  bad_date|        PayPal|     UK|             NULL|
|     145|       1700|       242|      Books|294.01|     8.0|     SUCCESS|  bad_date|    Debit Card|Vietnam|   

In [80]:
df_check_date.count()

160

In [81]:
df.filter(col("order_date") == "bad_date").count()

160

In [82]:
df.select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|     success|
|      failed|
|     Success|
|     SUCCESS|
|      FAILED|
|     PENDING|
+------------+



In [83]:
numeric_cols = ["order_id", "customer_id", "product_id","quantity"]
for c in numeric_cols:
    print(f"Checking column: {c}")
    
    df.filter(
        col(c).isNotNull() &
        col(c).cast("double").isNull()
    ).select(c).distinct().show(truncate=False)

Checking column: order_id
+--------+
|order_id|
+--------+
+--------+

Checking column: customer_id
+-----------+
|customer_id|
+-----------+
+-----------+

Checking column: product_id
+----------+
|product_id|
+----------+
+----------+

Checking column: quantity
+--------+
|quantity|
+--------+
+--------+



Processing Data

In [84]:
df = df.dropDuplicates(["order_id"])

In [85]:
df.count()

5000

In [86]:
df = df.filter(col("quantity").isNotNull())

In [87]:
df.count()

4484

In [88]:
df = df.filter(col("order_date") != "bad_date")

In [89]:
df.count()

4346

In [90]:
df = df.withColumn(
    "order_status",
    lower(trim(col("order_status")))
)

In [91]:
df.select("order_status").distinct().count()

3

In [92]:
df = df \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("order_date", to_date(col("order_date"),"yyyy-MM-dd"))

In [93]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country: string (nullable = true)



In [94]:
df = df.withColumn(
    "total_revenue",
    round(
        col("price")*col("quantity"),2
    )
)

In [95]:
df.show()

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-------------+
|order_id|customer_id|product_id|   category| price|quantity|order_status|order_date|payment_method|country|total_revenue|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-------------+
|       1|         52|       380|       Home|643.03|       3|     success|2026-03-12| Bank Transfer|     US|      1929.09|
|      10|        284|       261|     Sports|227.13|       1|      failed|2024-01-13| Bank Transfer| Canada|       227.13|
|     100|         29|       136|     Beauty|792.78|       6|     success|2024-08-09|        PayPal|     US|      4756.68|
|    1000|       1435|       319|     Sports|508.21|       4|     success|2023-07-12|        PayPal| France|      2032.84|
|    1001|         27|       342|     Sports|150.42|       1|     pending|2023-07-27|   Credit Card| Canada|       150.42|
|    1002|      

In [96]:
df.count()

4346

In [97]:
df.write.mode("append").parquet("/opt/spark/work-dir/data/ecommerce_data")